In [1]:
import requests, sys, json
import pandas as pd
import pymongo
import numpy as np

In [2]:
# Retrieve VEP objects 
def lookup_variant_effects(id_list):
 
    server = "https://grch37.rest.ensembl.org"
    ext = "/vep/human/id?canonical=1;CADD=1;vcf_string=1;SIFT=1;PolyPhen=1;domains=1;DisGeNET=1;EVE=1;GO=1;Mastermind=1;Phenotypes=1;mutfunc=1;variant_class=1;xref_refseq=1"
    
    headers={ "Content-Type" : "application/json", "Accept" : "application/json"}
    
    # Define the request body as a JSON string with the list of variant IDs
    data = {"ids": id_list}
    json_data = json.dumps(data) # the data will be in this format with the single quotations: '{"ids" : ["rs56116432", "COSM476" ] }'
    # Additional parameters for the VEP API


    r = requests.post(server+ext, headers=headers, data=json_data)

    if not r.ok:
      r.raise_for_status()
      sys.exit()

    decoded = r.json()
    
    return decoded
# print(type(repr(decoded)))

 

In [3]:
# load data
new = []

with open ("rsid_six_gene.txt", "r") as f:
    #entire content
    lines = f.readlines()

variant_ids = [line.strip() for line in lines]
len(variant_ids)

302

In [4]:
a = lookup_variant_effects(variant_ids[0:150])

In [5]:
b = lookup_variant_effects(variant_ids[151:302])

In [6]:
variant_effect = a + b 
df = pd.DataFrame(variant_effect )

In [7]:
df

,vcf_string,seq_region_name,input,most_severe_consequence,end,start,assembly_name,colocated_variants,allele_string,regulatory_feature_consequences,id,strand,transcript_consequences,variant_class,intergenic_consequences,motif_feature_consequences
0,5-148802811-A-C,5,rs353294,non_coding_transcript_exon_variant,148802811,148802811,GRCh37,"[{'end': 148802811, 'start': 148802811, 'stran...",A/C,"[{'variant_allele': 'C', 'cadd_phred': 3.093, ...",rs353294,1,"[{'gene_id': 'ENSG00000249669', 'biotype': 'li...",SNV,NaN,NaN
1,5-149001897-A-G,5,rs77489920,intron_variant,149001897,149001897,GRCh37,"[{'start': 149001897, 'end': 149001897, 'stran...",A/G,NaN,rs77489920,1,"[{'canonical': 1, 'cadd_phred': 6.479, 'varian...",SNV,NaN,NaN
2,5-149012782-G-A,5,rs3733658,3_prime_UTR_variant,149012782,149012782,GRCh37,"[{'minor_allele_freq': 0.2644, 'allele_string'...",G/A,"[{'cadd_raw': -0.044382, 'regulatory_feature_i...",rs3733658,1,"[{'cdna_end': 3219, 'cdna_start': 3219, 'gene_...",SNV,NaN,NaN
3,5-148981046-T-C,5,rs1379544,intron_variant,148981046,148981046,GRCh37,"[{'seq_region_name': '5', 'frequencies': {'C':...",T/C,NaN,rs1379544,1,"[{'consequence_terms': ['intron_variant'], 'hg...",SNV,NaN,NaN
4,5-148825669-C-T,5,rs10062536,intergenic_variant,148825669,148825669,GRCh37,"[{'seq_region_name': '5', 'minor_allele': 'T',...",C/T,NaN,rs10062536,1,NaN,SNV,"[{'variant_allele': 'T', 'cadd_raw': 0.34148, ...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,"[3-53282303-A-G, 3-53282303-A-T]",3,rs4687718,intron_variant,53282303,53282303,GRCh37,"[{'start': 53282303, 'allele_string': 'A/G/T',...",A/G/T,NaN,rs4687718,1,"[{'mastermind_counts': '35|35|79', 'consequenc...",SNV,NaN,NaN
319,"[HG957_PATCH-53282303-A-G, HG957_PATCH-5328230...",HG957_PATCH,rs4687718,intron_variant,53282303,53282303,GRCh37,"[{'end': 53282303, 'id': 'rs4687718', 'minor_a...",A/G/T,NaN,rs4687718,1,"[{'transcript_id': 'ENST00000608285', 'strand'...",SNV,NaN,NaN
320,3-53125585-T-C,3,rs2564921,3_prime_UTR_variant,53125585,53125585,GRCh37,"[{'var_synonyms': {'ClinVar': ['RCV000365605',...",T/C,NaN,rs2564921,1,"[{'canonical': 1, 'cadd_phred': 0.294, 'transc...",SNV,NaN,NaN
321,3-53140603-T-C,3,rs1004807,intron_variant,53140603,53140603,GRCh37,"[{'frequencies': {'C': {'eas': 0.8542, 'afr': ...",T/C,NaN,rs1004807,1,"[{'strand': -1, 'gene_symbol': 'RFT1', 'master...",SNV,NaN,NaN


* 278 SNPs in exon regions
* 45 SNPs in intronic regions
* 112 SNPs involved in regulatory 

In [31]:
323 - df["intergenic_consequences"].isna().sum()

45

In [32]:
323 - df["transcript_consequences"].isna().sum()

278

In [33]:
323- df["regulatory_feature_consequences"].isna().sum()

112

## The 6 Driving Genes
* PRKCD: ENSG00000163932 : 5580
* LRP5: ENSG00000162337 : 4041
* PPARD: ENSG00000112033 : 5467
* CSNK1A1: ENSG00000113712 : 1452
* AGT : ENSG00000135744 : 183
* PIK3R2 : ENSG00000105647 : 5296



In [7]:
### Capture Variants of all types of variants of these genes
name = ["ENSG00000163932", "ENSG00000162337", "ENSG00000112033", "ENSG00000113712", "ENSG00000135744","ENSG00000105647"]
for i in range(0,len(variant_effect)):
    list_intron = []
list_exon = []
name = ["ENSG00000163932", "ENSG00000162337", "ENSG00000112033", "ENSG00000113712", "ENSG00000135744","ENSG00000105647"]
for i in range(0,len(variant_effect)):
    if "intergenic_consequences" in variant_effect[i].keys():
        list_intron.append((df["id"][i],variant_effect[i]["intergenic_consequences"][0]))
        
    elif "transcript_consequences" in variant_effect[i].keys():
        for dic in variant_effect[i]["transcript_consequences"]:
            if dic["gene_id"] in name:
                list_exon.append((df["id"][i],dic))

complete_list_all = list_intron + list_exon  
len(complete_list_all)



519

In [8]:
complete_list_all

[('rs10062536',
  {'impact': 'MODIFIER',
   'cadd_phred': 7.734,
   'cadd_raw': 0.34148,
   'variant_allele': 'T',
   'consequence_terms': ['intergenic_variant']}),
 ('rs414582',
  {'impact': 'MODIFIER',
   'cadd_raw': -0.30287,
   'cadd_phred': 0.359,
   'consequence_terms': ['intergenic_variant'],
   'variant_allele': 'C'}),
 ('rs10061434',
  {'impact': 'MODIFIER',
   'cadd_raw': -0.059472,
   'cadd_phred': 1.828,
   'consequence_terms': ['intergenic_variant'],
   'variant_allele': 'T'}),
 ('rs10454990',
  {'impact': 'MODIFIER',
   'cadd_raw': 0.083336,
   'cadd_phred': 3.983,
   'phenotypes': [{'p_value': 5e-06,
     'end': 148838117,
     'odds_ratio': 4.558,
     'start': 148838117,
     'type': 'Variation',
     'strand': '+',
     'seq_region_name': '5',
     'source': 'NHGRI-EBI_GWAS_catalog',
     'id': 'rs10454990',
     'associated_gene': 'MIR143HG,CSNK1A1',
     'variation_names': 'rs10454990',
     'risk_allele': 'C',
     'phenotype': 'Carotid intima media thickness x smo

In [9]:
df_2_all = pd.DataFrame(complete_list_all, columns = ["id","consequences"])
expanded_df = df_2_all["consequences"].apply(pd.Series)
df_2_all = pd.concat([df_2_all, expanded_df], axis=1)

In [10]:
df_2_all

,id,consequences,impact,cadd_phred,cadd_raw,variant_allele,consequence_terms,phenotypes,gene_symbol,go,...,domains,cdna_start,protein_end,protein_start,sift_prediction,cdna_end,cds_start,codons,polyphen_prediction,sift_score
0,rs10062536,"{'impact': 'MODIFIER', 'cadd_phred': 7.734, 'c...",MODIFIER,7.734,0.341480,T,[intergenic_variant],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",MODIFIER,0.359,-0.302870,C,[intergenic_variant],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_raw': -0.059472, ...",MODIFIER,1.828,-0.059472,T,[intergenic_variant],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rs10454990,"{'impact': 'MODIFIER', 'cadd_raw': 0.083336, '...",MODIFIER,3.983,0.083336,G,[intergenic_variant],"[{'p_value': 5e-06, 'end': 148838117, 'odds_ra...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,rs72823546,"{'consequence_terms': ['intergenic_variant'], ...",MODIFIER,6.259,0.228896,A,[intergenic_variant],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,rs1483185,"{'cadd_raw': 0.154256, 'strand': 1, 'gene_symb...",MODIFIER,5.142,0.154256,G,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004699:calc...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
515,rs1483185,"{'flags': ['cds_end_NF'], 'variant_allele': 'A...",MODIFIER,5.268,0.162378,A,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004699:calc...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
516,rs1483185,"{'cadd_phred': 5.142, 'impact': 'MODIFIER', 'm...",MODIFIER,5.142,0.154256,G,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004699:calc...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
517,rs729396,"{'cadd_phred': 6.551, 'impact': 'MODIFIER', 'g...",MODIFIER,6.551,0.249872,T,[downstream_gene_variant],NaN,PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Index(['id', 'consequences', 'impact', 'cadd_phred', 'cadd_raw',
       'variant_allele', 'consequence_terms', 'phenotypes', 'gene_symbol',
       'go', 'transcript_id', 'hgnc_id', 'gene_symbol_source', 'biotype',
       'gene_id', 'strand', 'refseq_transcript_ids', 'flags', 'distance',
       'canonical', 'mastermind_counts', 'mastermind_mmid3', 'polyphen_score',
       'cds_end', 'amino_acids', 'domains', 'cdna_start', 'protein_end',
       'protein_start', 'sift_prediction', 'cdna_end', 'cds_start', 'codons',
       'polyphen_prediction', 'sift_score'],
      dtype='object')

In [12]:
merged_df_all = pd.merge(df_2_all, df, how="left", on="id")
merged_df_all.drop(columns=["input", "seq_region_name"])

,id,consequences,impact,cadd_phred,cadd_raw,variant_allele,consequence_terms,phenotypes,gene_symbol,go,...,vcf_string,colocated_variants,allele_string,strand_y,most_severe_consequence,transcript_consequences,start,variant_class,intergenic_consequences,motif_feature_consequences
0,rs10062536,"{'impact': 'MODIFIER', 'cadd_phred': 7.734, 'c...",MODIFIER,7.734,0.341480,T,[intergenic_variant],NaN,NaN,NaN,...,5-148825669-C-T,"[{'strand': 1, 'id': 'rs10062536', 'seq_region...",C/T,1,intergenic_variant,NaN,148825669,SNV,"[{'impact': 'MODIFIER', 'cadd_phred': 7.734, '...",NaN
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",MODIFIER,0.359,-0.302870,C,[intergenic_variant],NaN,NaN,NaN,...,5-148851149-T-C,"[{'minor_allele_freq': 0.1869, 'minor_allele':...",T/C,1,regulatory_region_variant,NaN,148851149,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': -0.30287, ...",NaN
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_raw': -0.059472, ...",MODIFIER,1.828,-0.059472,T,[intergenic_variant],NaN,NaN,NaN,...,5-149023879-C-T,"[{'start': 149023879, 'id': 'rs10061434', 'all...",C/T,1,intergenic_variant,NaN,149023879,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': -0.059472,...",NaN
3,rs10454990,"{'impact': 'MODIFIER', 'cadd_raw': 0.083336, '...",MODIFIER,3.983,0.083336,G,[intergenic_variant],"[{'p_value': 5e-06, 'end': 148838117, 'odds_ra...",NaN,NaN,...,"[5-148838117-C-G, 5-148838117-C-T]","[{'minor_allele_freq': 0.356, 'phenotype_or_di...",C/G/T,1,regulatory_region_variant,NaN,148838117,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': 0.083336, ...",NaN
4,rs72823546,"{'consequence_terms': ['intergenic_variant'], ...",MODIFIER,6.259,0.228896,A,[intergenic_variant],NaN,NaN,NaN,...,"[5-148850866-G-A, 5-148850866-G-C]","[{'strand': 1, 'seq_region_name': '5', 'allele...",G/A/C,1,regulatory_region_variant,NaN,148850866,SNV,"[{'consequence_terms': ['intergenic_variant'],...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,rs1483185,"{'cadd_phred': 5.142, 'impact': 'MODIFIER', 'm...",MODIFIER,5.142,0.154256,G,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004699:calc...",...,"[HG957_PATCH-53199014-T-A, HG957_PATCH-5319901...","[{'start': 53199014, 'seq_region_name': 'HG957...",T/A/G,1,intron_variant,"[{'hgnc_id': 9399, 'gene_symbol': 'PRKCD', 'va...",53199014,SNV,NaN,NaN
603,rs729396,"{'cadd_phred': 6.551, 'impact': 'MODIFIER', 'g...",MODIFIER,6.551,0.249872,T,[downstream_gene_variant],NaN,PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,3-53228342-C-T,"[{'start': 53228342, 'seq_region_name': '3', '...",C/T,1,downstream_gene_variant,"[{'cadd_phred': 6.551, 'impact': 'MODIFIER', '...",53228342,SNV,NaN,NaN
604,rs729396,"{'cadd_phred': 6.551, 'impact': 'MODIFIER', 'g...",MODIFIER,6.551,0.249872,T,[downstream_gene_variant],NaN,PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,HG957_PATCH-53228342-C-T,"[{'minor_allele_freq': 0.1857, 'id': 'rs729396...",C/T,1,downstream_gene_variant,"[{'variant_allele': 'T', 'gene_symbol': 'PRKCD...",53228342,SNV,NaN,NaN
605,rs729396,"{'cadd_phred': 6.551, 'impact': 'MODIFIER', 'g...",MODIFIER,6.551,0.249872,T,[downstream_gene_variant],NaN,PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,3-53228342-C-T,"[{'start': 53228342, 'seq_region_name': '3', '...",C/T,1,downstream_gene_variant,"[{'cadd_phred': 6.551, 'impact': 'MODIFIER', '...",53228342,SNV,NaN,NaN


In [38]:
df_uniqued_all = merged_df_all.drop_duplicates(subset=['id', 'cadd_phred', 'variant_allele'])

In [39]:
df_uniqued_all

,id,consequences,impact,cadd_phred,cadd_raw,variant_allele,consequence_terms,phenotypes,gene_symbol,go,...,allele_string,seq_region_name,input,strand_y,most_severe_consequence,transcript_consequences,start,variant_class,intergenic_consequences,motif_feature_consequences
0,rs10062536,"{'impact': 'MODIFIER', 'cadd_phred': 7.734, 'c...",MODIFIER,7.734,0.341480,T,[intergenic_variant],NaN,NaN,NaN,...,C/T,5,rs10062536,1,intergenic_variant,NaN,148825669,SNV,"[{'impact': 'MODIFIER', 'cadd_phred': 7.734, '...",NaN
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",MODIFIER,0.359,-0.302870,C,[intergenic_variant],NaN,NaN,NaN,...,T/C,5,rs414582,1,regulatory_region_variant,NaN,148851149,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': -0.30287, ...",NaN
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_raw': -0.059472, ...",MODIFIER,1.828,-0.059472,T,[intergenic_variant],NaN,NaN,NaN,...,C/T,5,rs10061434,1,intergenic_variant,NaN,149023879,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': -0.059472,...",NaN
3,rs10454990,"{'impact': 'MODIFIER', 'cadd_raw': 0.083336, '...",MODIFIER,3.983,0.083336,G,[intergenic_variant],"[{'p_value': 5e-06, 'end': 148838117, 'odds_ra...",NaN,NaN,...,C/G/T,5,rs10454990,1,regulatory_region_variant,NaN,148838117,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': 0.083336, ...",NaN
4,rs72823546,"{'consequence_terms': ['intergenic_variant'], ...",MODIFIER,6.259,0.228896,A,[intergenic_variant],NaN,NaN,NaN,...,G/A/C,5,rs72823546,1,regulatory_region_variant,NaN,148850866,SNV,"[{'consequence_terms': ['intergenic_variant'],...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,rs11924565,"{'cadd_phred': 5.401, 'impact': 'MODIFIER', 'm...",MODIFIER,5.401,0.170975,G,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,C/A/G/T,3,rs11924565,1,intron_variant,"[{'consequence_terms': ['intron_variant'], 'va...",53197640,SNV,NaN,[{'consequence_terms': ['TF_binding_site_varia...
547,rs11924565,"{'strand': 1, 'gene_symbol_source': 'HGNC', 't...",MODIFIER,6.272,0.229776,T,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,C/A/G/T,3,rs11924565,1,intron_variant,"[{'consequence_terms': ['intron_variant'], 'va...",53197640,SNV,NaN,[{'consequence_terms': ['TF_binding_site_varia...
579,rs1483185,"{'impact': 'MODIFIER', 'cadd_phred': 5.268, 'g...",MODIFIER,5.268,0.162378,A,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,T/A/G,3,rs1483185,1,intron_variant,"[{'impact': 'MODIFIER', 'cadd_phred': 5.268, '...",53199014,SNV,NaN,NaN
581,rs1483185,"{'gene_symbol_source': 'HGNC', 'strand': 1, 't...",MODIFIER,5.142,0.154256,G,[intron_variant],"[{'seq_region_name': '3', 'end': 53226733, 'st...",PRKCD,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,T/A/G,3,rs1483185,1,intron_variant,"[{'impact': 'MODIFIER', 'cadd_phred': 5.268, '...",53199014,SNV,NaN,NaN


In [40]:
df_uniqued_all.to_csv("./feature_tsv/VEP_noncanonical_included_Output.tsv", sep = '\t', index= False)


Only contain variants from Canoncical transcripts for variants in exons

In [9]:
list_intron = []
list_exon = []
name = ["ENSG00000163932", "ENSG00000162337", "ENSG00000112033", "ENSG00000113712", "ENSG00000135744","ENSG00000105647"]
for i in range(0,len(variant_effect)):
    if "intergenic_consequences" in variant_effect[i].keys():
        list_intron.append((df["id"][i],variant_effect[i]["intergenic_consequences"][0]))
        
    elif "transcript_consequences" in variant_effect[i].keys():
        for dic in variant_effect[i]["transcript_consequences"]:
            if "canonical" in dic and dic["gene_id"] in name:
                list_exon.append((df["id"][i],dic))

complete_list = list_intron + list_exon  
len(complete_list)


153

In [10]:
complete_list

[('rs10062536',
  {'variant_allele': 'T',
   'cadd_raw': 0.34148,
   'cadd_phred': 7.734,
   'impact': 'MODIFIER',
   'consequence_terms': ['intergenic_variant']}),
 ('rs414582',
  {'consequence_terms': ['intergenic_variant'],
   'impact': 'MODIFIER',
   'variant_allele': 'C',
   'cadd_raw': -0.30287,
   'cadd_phred': 0.359}),
 ('rs10061434',
  {'impact': 'MODIFIER',
   'consequence_terms': ['intergenic_variant'],
   'variant_allele': 'T',
   'cadd_phred': 1.828,
   'cadd_raw': -0.059472}),
 ('rs10454990',
  {'impact': 'MODIFIER',
   'consequence_terms': ['intergenic_variant'],
   'variant_allele': 'G',
   'cadd_phred': 3.983,
   'cadd_raw': 0.083336,
   'phenotypes': [{'variation_names': 'rs10454990',
     'odds_ratio': 4.558,
     'p_value': 5e-06,
     'seq_region_name': '5',
     'risk_allele': 'C',
     'id': 'rs10454990',
     'type': 'Variation',
     'associated_gene': 'MIR143HG,CSNK1A1',
     'start': 148838117,
     'source': 'NHGRI-EBI_GWAS_catalog',
     'phenotype': 'Carot

In [63]:
df_2 = pd.DataFrame(complete_list, columns = ["id","consequences"])
expanded_df = df_2["consequences"].apply(pd.Series)
df_2 = pd.concat([df_2, expanded_df], axis=1)

In [64]:
df_2.columns

Index(['id', 'consequences', 'cadd_raw', 'variant_allele', 'cadd_phred',
       'consequence_terms', 'impact', 'phenotypes', 'biotype', 'gene_id', 'go',
       'transcript_id', 'canonical', 'strand', 'gene_symbol_source',
       'gene_symbol', 'hgnc_id', 'mastermind_counts', 'mastermind_mmid3',
       'refseq_transcript_ids', 'polyphen_score', 'cds_end', 'sift_prediction',
       'protein_end', 'amino_acids', 'domains', 'codons', 'cdna_start',
       'protein_start', 'cds_start', 'cdna_end', 'sift_score',
       'polyphen_prediction', 'distance'],
      dtype='object')

In [66]:
df_2

,id,consequences,cadd_raw,variant_allele,cadd_phred,consequence_terms,impact,phenotypes,biotype,gene_id,...,amino_acids,domains,codons,cdna_start,protein_start,cds_start,cdna_end,sift_score,polyphen_prediction,distance
0,rs10062536,"{'cadd_raw': 0.34148, 'variant_allele': 'T', '...",0.341480,T,7.734,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",-0.302870,C,0.359,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_phred': 1.828, 'v...",-0.059472,T,1.828,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rs10454990,"{'phenotypes': [{'end': 148838117, 'p_value': ...",0.083336,G,3.983,[intergenic_variant],MODIFIER,"[{'end': 148838117, 'p_value': 5e-06, 'seq_reg...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,rs72823546,"{'variant_allele': 'A', 'cadd_phred': 6.259, '...",0.228896,A,6.259,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,rs11924565,"{'biotype': 'protein_coding', 'consequence_ter...",0.170975,G,5.401,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,rs11924565,"{'consequence_terms': ['intron_variant'], 'bio...",0.229776,T,6.272,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,rs1483185,"{'mastermind_mmid3': 'PRKCD:5UTR,PRKCD:5UTRint...",0.162378,A,5.268,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,rs1483185,"{'gene_symbol_source': 'HGNC', 'variant_allele...",0.154256,G,5.142,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
# merged_df = pd.merge(df_2, df, how="left", on="id")
# merged_df.drop(columns=["input", "seq_region_name"])

,id,consequences,cadd_raw,variant_allele,cadd_phred,consequence_terms,impact,phenotypes,biotype,gene_id,...,colocated_variants,most_severe_consequence,vcf_string,end,assembly_name,regulatory_feature_consequences,allele_string,variant_class,intergenic_consequences,motif_feature_consequences
0,rs10062536,"{'cadd_raw': 0.34148, 'variant_allele': 'T', '...",0.341480,T,7.734,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,"[{'minor_allele_freq': 0.1953, 'strand': 1, 'e...",intergenic_variant,5-148825669-C-T,148825669,GRCh37,NaN,C/T,SNV,"[{'cadd_raw': 0.34148, 'variant_allele': 'T', ...",NaN
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",-0.302870,C,0.359,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,"[{'start': 148851149, 'minor_allele': 'C', 'id...",regulatory_region_variant,5-148851149-T-C,148851149,GRCh37,[{'consequence_terms': ['regulatory_region_var...,T/C,SNV,"[{'impact': 'MODIFIER', 'cadd_raw': -0.30287, ...",NaN
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_phred': 1.828, 'v...",-0.059472,T,1.828,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,"[{'allele_string': 'C/T', 'minor_allele_freq':...",intergenic_variant,5-149023879-C-T,149023879,GRCh37,NaN,C/T,SNV,"[{'impact': 'MODIFIER', 'cadd_phred': 1.828, '...",NaN
3,rs10454990,"{'phenotypes': [{'end': 148838117, 'p_value': ...",0.083336,G,3.983,[intergenic_variant],MODIFIER,"[{'end': 148838117, 'p_value': 5e-06, 'seq_reg...",NaN,NaN,...,"[{'phenotype_or_disease': 1, 'pubmed': [321174...",regulatory_region_variant,"[5-148838117-C-G, 5-148838117-C-T]",148838117,GRCh37,[{'consequence_terms': ['regulatory_region_var...,C/G/T,SNV,"[{'phenotypes': [{'end': 148838117, 'p_value':...",NaN
4,rs72823546,"{'variant_allele': 'A', 'cadd_phred': 6.259, '...",0.228896,A,6.259,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,"[{'allele_string': 'G/A/C', 'end': 148850866, ...",regulatory_region_variant,"[5-148850866-G-A, 5-148850866-G-C]",148850866,GRCh37,"[{'regulatory_feature_id': 'ENSR00000188817', ...",G/A/C,SNV,"[{'variant_allele': 'A', 'cadd_phred': 6.259, ...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,rs1483185,"{'mastermind_mmid3': 'PRKCD:5UTR,PRKCD:5UTRint...",0.162378,A,5.268,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,"[{'allele_string': 'T/A/G', 'strand': 1, 'seq_...",intron_variant,"[HG957_PATCH-53199014-T-A, HG957_PATCH-5319901...",53199014,GRCh37,NaN,T/A/G,SNV,NaN,NaN
170,rs1483185,"{'gene_symbol_source': 'HGNC', 'variant_allele...",0.154256,G,5.142,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,"[{'id': 'rs1483185', 'start': 53199014, 'pubme...",intron_variant,"[3-53199014-T-A, 3-53199014-T-G]",53199014,GRCh37,"[{'biotype': 'CTCF_binding_site', 'cadd_phred'...",T/A/G,SNV,NaN,NaN
171,rs1483185,"{'gene_symbol_source': 'HGNC', 'variant_allele...",0.154256,G,5.142,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,"[{'allele_string': 'T/A/G', 'strand': 1, 'seq_...",intron_variant,"[HG957_PATCH-53199014-T-A, HG957_PATCH-5319901...",53199014,GRCh37,NaN,T/A/G,SNV,NaN,NaN
172,rs729396,"{'go': 'GO:0001666:response_to_hypoxia,GO:0004...",0.249872,T,6.551,[downstream_gene_variant],MODIFIER,NaN,protein_coding,ENSG00000163932,...,"[{'allele_string': 'C/T', 'frequencies': {'T':...",downstream_gene_variant,3-53228342-C-T,53228342,GRCh37,NaN,C/T,SNV,NaN,NaN


In [16]:
df_unique = df[~df['id'].duplicated(keep="first")]
df_unique = df_unique[["id","allele_string", "vcf_string"]]

In [17]:
df_unique

,id,allele_string,vcf_string
0,rs353294,A/C,5-148802811-A-C
1,rs77489920,A/G,5-149001897-A-G
2,rs3733658,G/A,5-149012782-G-A
3,rs1379544,T/C,5-148981046-T-C
4,rs10062536,C/T,5-148825669-C-T
...,...,...,...
317,rs75323042,T/C/G,"[3-53139644-T-C, 3-53139644-T-G]"
318,rs4687718,A/G/T,"[3-53282303-A-G, 3-53282303-A-T]"
320,rs2564921,T/C,3-53125585-T-C
321,rs1004807,T/C,3-53140603-T-C


In [156]:
vep = pd.merge(df_2, df_unique, on = "id", how="left")

In [157]:
vep

,id,consequences,cadd_raw,variant_allele,cadd_phred,consequence_terms,impact,phenotypes,biotype,gene_id,...,codons,cdna_start,protein_start,cds_start,cdna_end,sift_score,polyphen_prediction,distance,allele_string,vcf_string
0,rs10062536,"{'cadd_raw': 0.34148, 'variant_allele': 'T', '...",0.341480,T,7.734,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/T,5-148825669-C-T
1,rs414582,"{'impact': 'MODIFIER', 'cadd_raw': -0.30287, '...",-0.302870,C,0.359,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/C,5-148851149-T-C
2,rs10061434,"{'impact': 'MODIFIER', 'cadd_phred': 1.828, 'v...",-0.059472,T,1.828,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/T,5-149023879-C-T
3,rs10454990,"{'phenotypes': [{'end': 148838117, 'p_value': ...",0.083336,G,3.983,[intergenic_variant],MODIFIER,"[{'end': 148838117, 'p_value': 5e-06, 'seq_reg...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/G/T,"[5-148838117-C-G, 5-148838117-C-T]"
4,rs72823546,"{'variant_allele': 'A', 'cadd_phred': 6.259, '...",0.228896,A,6.259,[intergenic_variant],MODIFIER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,G/A/C,"[5-148850866-G-A, 5-148850866-G-C]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,rs11924565,"{'biotype': 'protein_coding', 'consequence_ter...",0.170975,G,5.401,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/A/G/T,"[3-53197640-C-A, 3-53197640-C-G, 3-53197640-C-T]"
149,rs11924565,"{'consequence_terms': ['intron_variant'], 'bio...",0.229776,T,6.272,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/A/G/T,"[3-53197640-C-A, 3-53197640-C-G, 3-53197640-C-T]"
150,rs1483185,"{'mastermind_mmid3': 'PRKCD:5UTR,PRKCD:5UTRint...",0.162378,A,5.268,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/A/G,"[3-53199014-T-A, 3-53199014-T-G]"
151,rs1483185,"{'gene_symbol_source': 'HGNC', 'variant_allele...",0.154256,G,5.142,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/A/G,"[3-53199014-T-A, 3-53199014-T-G]"


In [158]:
vep["variant"] = vep["allele_string"].str[0] + '/' + vep["variant_allele"]
vep = vep.drop(columns=["consequences"])
vep

,id,cadd_raw,variant_allele,cadd_phred,consequence_terms,impact,phenotypes,biotype,gene_id,go,...,cdna_start,protein_start,cds_start,cdna_end,sift_score,polyphen_prediction,distance,allele_string,vcf_string,variant
0,rs10062536,0.341480,T,7.734,[intergenic_variant],MODIFIER,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/T,5-148825669-C-T,C/T
1,rs414582,-0.302870,C,0.359,[intergenic_variant],MODIFIER,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/C,5-148851149-T-C,T/C
2,rs10061434,-0.059472,T,1.828,[intergenic_variant],MODIFIER,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/T,5-149023879-C-T,C/T
3,rs10454990,0.083336,G,3.983,[intergenic_variant],MODIFIER,"[{'end': 148838117, 'p_value': 5e-06, 'seq_reg...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/G/T,"[5-148838117-C-G, 5-148838117-C-T]",C/G
4,rs72823546,0.228896,A,6.259,[intergenic_variant],MODIFIER,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,G/A/C,"[5-148850866-G-A, 5-148850866-G-C]",G/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,rs11924565,0.170975,G,5.401,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/A/G/T,"[3-53197640-C-A, 3-53197640-C-G, 3-53197640-C-T]",C/G
149,rs11924565,0.229776,T,6.272,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C/A/G/T,"[3-53197640-C-A, 3-53197640-C-G, 3-53197640-C-T]",C/T
150,rs1483185,0.162378,A,5.268,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/A/G,"[3-53199014-T-A, 3-53199014-T-G]",T/A
151,rs1483185,0.154256,G,5.142,[intron_variant],MODIFIER,"[{'strand': '+', 'seq_region_name': '3', 'end'...",protein_coding,ENSG00000163932,"GO:0001666:response_to_hypoxia,GO:0004672:prot...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T/A/G,"[3-53199014-T-A, 3-53199014-T-G]",T/G


In [ ]:
vep["chr"] = vep["vcf_string"].astype(str).str[0]
type(vep["chr"])

In [150]:
for chrom, group in vep.groupby("chr"):
    print(chrom,":",group["id"].tolist())

1 : ['rs118050745', 'rs113800014', 'rs11602367', 'rs474742', 'rs853469', 'rs77645389', 'rs62120362', 'rs599083', 'rs72932379', 'rs111489476', 'rs11826287', 'rs3781590', 'rs118062233', 'rs35388122', 'rs632605', 'rs118098503', 'rs76710454', 'rs3736228', 'rs75177427', 'rs11574424', 'rs4988300', 'rs3781586', 'rs12417014', 'rs4988332', 'rs75210709', 'rs2306862', 'rs943581', 'rs2067853', 'rs5049', 'rs699', 'rs11122577', 'rs11568048', 'rs7079', 'rs10864772', 'rs2493131', 'rs4762', 'rs3730179', 'rs11554159']
3 : ['rs115657945', 'rs79843306', 'rs79843306', 'rs116071986', 'rs1080500', 'rs750527', 'rs73091425', 'rs11130350', 'rs114058530', 'rs73076230', 'rs729396']
5 : ['rs10062536', 'rs414582', 'rs10061434', 'rs4705346', 'rs11745684', 'rs78578633', 'rs17724303', 'rs72823547', 'rs73279543', 'rs452366', 'rs241280', 'rs62378023', 'rs2125130', 'rs111416523', 'rs114688286']
6 : ['rs6457821', 'rs115269427', 'rs78945013', 'rs2267665', 'rs115128769', 'rs116213531', 'rs9462081', 'rs7770619', 'rs6937510',

In [153]:
l = ['rs118050745', 'rs113800014', 'rs11602367', 'rs474742', 'rs853469', 'rs77645389', 'rs62120362', 'rs599083', 'rs72932379', 'rs111489476', 'rs11826287', 'rs3781590', 'rs118062233', 'rs35388122', 'rs632605', 'rs118098503', 'rs76710454', 'rs3736228', 'rs75177427', 'rs11574424', 'rs4988300', 'rs3781586', 'rs12417014', 'rs4988332', 'rs75210709', 'rs2306862', 'rs943581', 'rs2067853', 'rs5049', 'rs699', 'rs11122577', 'rs11568048', 'rs7079', 'rs10864772', 'rs2493131', 'rs4762', 'rs3730179', 'rs11554159']
for i in l:
    print (i)

rs118050745
rs113800014
rs11602367
rs474742
rs853469
rs77645389
rs62120362
rs599083
rs72932379
rs111489476
rs11826287
rs3781590
rs118062233
rs35388122
rs632605
rs118098503
rs76710454
rs3736228
rs75177427
rs11574424
rs4988300
rs3781586
rs12417014
rs4988332
rs75210709
rs2306862
rs943581
rs2067853
rs5049
rs699
rs11122577
rs11568048
rs7079
rs10864772
rs2493131
rs4762
rs3730179
rs11554159


In [121]:
print("\n".join(vep["id"].astype(str)))

rs10062536
rs414582
rs10061434
rs10454990
rs72823546
rs353237
rs4705346
rs11745684
rs2454104
rs78578633
rs17724303
rs447950
rs72823547
rs73279543
rs452366
rs241280
rs353248
rs353243
rs353241
rs62378023
rs118050745
rs113800014
rs11602367
rs474742
rs4713858
rs6457821
rs115269427
rs78945013
rs853469
rs12031571
rs77645389
rs4846997
rs11122563
rs11122562
rs62120362
rs115659375
rs115659375
rs11919522
rs11919522
rs115657945
rs79843306
rs79843306
rs116071986
rs7610701
rs1080500
rs2125130
rs111416523
rs114688286
rs599083
rs72932379
rs111489476
rs11826287
rs3781590
rs660925
rs660925
rs660925
rs118062233
rs57928864
rs57928864
rs4988321
rs4988321
rs35388122
rs632605
rs118098503
rs312009
rs312009
rs76710454
rs3736228
rs75177427
rs11574424
rs4988300
rs3781586
rs634008
rs634008
rs634008
rs12417014
rs314750
rs314750
rs4988332
rs75210709
rs2306862
rs1883322
rs1883322
rs1883322
rs2267665
rs2016520
rs2016520
rs115128769
rs116213531
rs4713854
rs4713854
rs1003973
rs1003973
rs1003973
rs9462081
rs7770619
rs7

In [108]:
vep.to_csv("VEP_Output.tsv", sep = '\t', index= False)

In [15]:
myclient = pymongo.MongoClient("mongodb://localhost:27017")
db = myclient["variantDatabase"]
col = db["transcript_consequences"]